In [1]:
import re
import pandas as pd
from pathlib import Path

In [2]:
INPUT_PATH = Path("data\cleaned_job_data.csv")
OUTPUT_PATH = Path("data/cleaned_job_data_plus.csv")

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"ไม่พบไฟล์: {INPUT_PATH}")

df = pd.read_csv(INPUT_PATH)
print("โหลดข้อมูลสำเร็จ")
print("Shape:", df.shape)
print(df.head())

โหลดข้อมูลสำเร็จ
Shape: (1095, 6)
           Category Workplace                                      Location  \
0  Business Analyst    Remote                                United Kingdom   
1  Business Analyst    Remote             Makati, Metro Manila, Philippines   
2  Business Analyst   On-site  Al-Dajeej, Al Farwaniyah Governorate, Kuwait   
3  Business Analyst   On-site               London, England, United Kingdom   
4  Business Analyst    Remote                                United Kingdom   

              Department       Type  \
0             Operations  Full time   
1                 Aux HQ  Full time   
2       PWC Technologies  Full time   
3  Consultants, Advisory  Full time   
4             Operations  Full time   

                                      final_features  
0  business analyst remote united kingdom operati...  
1  business analyst remote makati metro manila ph...  
2  business analyst on site al dajeej al farwaniy...  
3  business analyst on site london e

<>:1: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<>:1: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
C:\Users\note3\AppData\Local\Temp\ipykernel_5676\2628849012.py:1: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
  INPUT_PATH = Path("data\cleaned_job_data.csv")


In [3]:
def clean_text(text):
    if pd.isna(text) or str(text).strip() == "":
        return "unknown"
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9+#\s]", " ", text)
    text = " ".join(text.split())
    return text if text else "unknown"


for col in ["Category", "Workplace", "Location", "Department", "Type", "final_features"]:
    if col not in df.columns:
        df[col] = "unknown"
    df[col] = df[col].fillna("unknown").apply(clean_text)

In [4]:
#สร้าง Skills แบบ rule-based
skill_rules = {
    "Business Analyst": [
        "excel sql powerpoint dashboard reporting business analysis stakeholder communication"
    ],
    "Cloud": [
        "aws azure gcp docker kubernetes linux networking devops cloud security terraform"
    ],
    "Data Scientist": [
        "python sql pandas numpy scikit learn machine learning statistics data visualization deep learning"
    ],
    "HR": [
        "recruitment onboarding payroll employee relations hr communication talent acquisition people management"
    ],
    "Software Developer": [
        "python java javascript c++ sql git api backend frontend fastapi django flask react nodejs"
    ],
    "UI/UX": [
        "figma wireframe prototype user research usability design system ui ux visual design"
    ]
}

def infer_skills(row):
    category = str(row["Category"]).strip()
    text = f'{row["Category"]} {row["Department"]} {row["Type"]} {row["final_features"]}'.lower()

    if category in skill_rules:
        return skill_rules[category][0]

    # fallback rules
    if "cloud" in text:
        return "aws azure gcp docker kubernetes linux devops networking"
    elif "data" in text or "scientist" in text:
        return "python sql pandas numpy machine learning statistics data visualization"
    elif "hr" in text:
        return "recruitment onboarding payroll communication people management"
    elif "ui" in text or "ux" in text or "design" in text:
        return "figma wireframe prototype user research visual design"
    elif "developer" in text or "software" in text:
        return "python java javascript sql git api backend frontend"
    elif "analyst" in text:
        return "excel sql reporting dashboard business analysis communication"
    else:
        return "communication teamwork problem solving"
        

df["Skills"] = df.apply(infer_skills, axis=1)
df["Skills"] = df["Skills"].apply(clean_text)

print(df[["Category", "Skills"]].head(10))

           Category                                             Skills
0  business analyst  excel sql reporting dashboard business analysi...
1  business analyst  figma wireframe prototype user research visual...
2  business analyst  excel sql reporting dashboard business analysi...
3  business analyst  excel sql reporting dashboard business analysi...
4  business analyst  excel sql reporting dashboard business analysi...
5  business analyst  figma wireframe prototype user research visual...
6  business analyst  excel sql reporting dashboard business analysi...
7  business analyst  excel sql reporting dashboard business analysi...
8  business analyst  excel sql reporting dashboard business analysi...
9  business analyst  figma wireframe prototype user research visual...


In [5]:
#สร้าง Keywords จากข้อความเดิม
def infer_keywords(row):
    parts = [
        str(row["Category"]),
        str(row["Department"]),
        str(row["Type"]),
        str(row["Location"]),
        str(row["Workplace"])
    ]
    text = " ".join(parts).lower()
    text = clean_text(text)
    return text

df["Keywords"] = df.apply(infer_keywords, axis=1)
print(df[["Category", "Keywords"]].head(10))

           Category                                           Keywords
0  business analyst  business analyst operations full time united k...
1  business analyst  business analyst aux hq full time makati metro...
2  business analyst  business analyst pwc technologies full time al...
3  business analyst  business analyst consultants advisory full tim...
4  business analyst  business analyst operations full time united k...
5  business analyst  business analyst aux hq full time makati metro...
6  business analyst  business analyst pwc technologies full time al...
7  business analyst  business analyst consultants advisory full tim...
8  business analyst  business analyst change innovation unknown lon...
9  business analyst  business analyst aux hq full time makati metro...


In [6]:
#สร้าง Description แบบ template
def infer_description(row):
    category = row["Category"]
    dept = row["Department"]
    job_type = row["Type"]
    location = row["Location"]
    workplace = row["Workplace"]
    skills = row["Skills"]

    desc = (
        f"{category} role in {dept} department. "
        f"This is a {job_type} position based in {location} with workplace style {workplace}. "
        f"Preferred skills include {skills}. "
        f"The role requires communication teamwork and problem solving."
    )
    return clean_text(desc)

df["Description"] = df.apply(infer_description, axis=1)
print(df[["Category", "Description"]].head(5))

           Category                                        Description
0  business analyst  business analyst role in operations department...
1  business analyst  business analyst role in aux hq department thi...
2  business analyst  business analyst role in pwc technologies depa...
3  business analyst  business analyst role in consultants advisory ...
4  business analyst  business analyst role in operations department...


In [7]:
#สร้าง Experience แบบ rule-based
def infer_experience(row):
    text = f'{row["Category"]} {row["Type"]} {row["Department"]} {row["final_features"]}'.lower()

    if any(k in text for k in ["intern", "junior", "entry", "fresher"]):
        return "junior"
    elif any(k in text for k in ["senior", "lead", "principal"]):
        return "senior"
    elif any(k in text for k in ["manager", "head", "director"]):
        return "manager"
    else:
        # ถ้าไม่มี clue ให้กลาง ๆ ไว้ก่อน
        return "mid"

df["Experience"] = df.apply(infer_experience, axis=1)
print(df[["Category", "Experience"]].head(10))

           Category Experience
0  business analyst        mid
1  business analyst        mid
2  business analyst        mid
3  business analyst        mid
4  business analyst        mid
5  business analyst        mid
6  business analyst        mid
7  business analyst        mid
8  business analyst        mid
9  business analyst        mid


In [8]:
#final_features_plus
for col in ["Skills", "Keywords", "Description", "Experience"]:
    df[col] = df[col].fillna("unknown").apply(clean_text)

df["final_features_plus"] = (
    df["final_features"].astype(str) + " " +
    df["Skills"].astype(str) + " " +
    df["Keywords"].astype(str) + " " +
    df["Description"].astype(str) + " " +
    df["Experience"].astype(str)
)

df["final_features_plus"] = df["final_features_plus"].apply(clean_text)

print(df[[
    "Category",
    "Skills",
    "Keywords",
    "Description",
    "Experience",
    "final_features_plus"
]].head(5))

           Category                                             Skills  \
0  business analyst  excel sql reporting dashboard business analysi...   
1  business analyst  figma wireframe prototype user research visual...   
2  business analyst  excel sql reporting dashboard business analysi...   
3  business analyst  excel sql reporting dashboard business analysi...   
4  business analyst  excel sql reporting dashboard business analysi...   

                                            Keywords  \
0  business analyst operations full time united k...   
1  business analyst aux hq full time makati metro...   
2  business analyst pwc technologies full time al...   
3  business analyst consultants advisory full tim...   
4  business analyst operations full time united k...   

                                         Description Experience  \
0  business analyst role in operations department...        mid   
1  business analyst role in aux hq department thi...        mid   
2  business analy

In [10]:
#save file
cols_to_save = [
    "Category", "Workplace", "Location", "Department", "Type",
    "final_features", "Skills", "Keywords", "Description", "Experience",
    "final_features_plus"
]

df[cols_to_save].to_csv(OUTPUT_PATH, index=False)
print(f"บันทึกไฟล์สำเร็จ: {OUTPUT_PATH}")
print("Shape:", df[cols_to_save].shape)

บันทึกไฟล์สำเร็จ: data\cleaned_job_data_plus.csv
Shape: (1095, 11)
